# Processing CESM2-LENS PiC data

### Set up
#### Packages

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from scipy import stats
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from Processing_functions import FixLongitude, FixTime, FixGrid, InterPlevels
xr.set_options(display_expand_data=False);
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import random
import os

#### Filepaths & name variables

In [ ]:
## File name
cesm2piC = 'b.e21.B1850.f09_g17.CMIP6-piControl.001'

## Filepaths
path_to_arch = "/glade/campaign/collections/cmip/CMIP6/timeseries-cmip6/"
comp = 'atm'
freq = 0 # 0: monthly, 1: daily
var_ind = 0

# ATM
# 0,     1     
# TREFHT, RESTOM

# ICE
# 0,    1 
# aice, hi

# OCN
# 0
# MOC

# Variables
var_list = {'atm': ['TREFHT','RESTOM'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]

# Extensions
h_ext = {'atm': ['.h0.'],
       'ice': ['.h.','.h1.'],
       'ocn': ['.h.']}
mod_com = {'atm': 'cam',
           'ice': 'cice',
           'ocn': 'pop'}
time_path = {'atm': ['month_1'],
                'ice': ['month_1','day_1'],
                'ocn': ['month_1']}
vert_lev = {'atm': [False,False],
            'ice': [False,False],
            'ocn': [False]}

path_to_outdata = '/glade/work/glydia/processed_CESM2_LENS_data/longer_recent/'
plot_levels = [300,500,850,925]

In [ ]:
cluster = PBSCluster(cores    = 1,
                     memory   = '25GiB',
                     queue    = 'casper',
                     walltime = '04:00:00',
                     account  = 'UCUB0137',
                     log_directory = '/glade/u/home/glydia/python/logs/',
                     name='piControl_'+var)
cluster.scale(4*9)
client = Client(cluster)

In [ ]:
client

In [ ]:
## Chunking variables
time_ch = 365*2 if freq == 1 else 600
chunks = {
    'atm': {'time': time_ch, 'lat': 96, 'lon': 144, 'lev': -1},
    'ice': {'time': time_ch, 'nj': 192, 'ni': 160},
    'ocn': {'time': time_ch, 'nlat': 64, 'nlon': 96, 'z_t': 5}
}

### Load & modify data
#### Control data

In [ ]:
startyrs = np.array([501,
 797,
 1328,
 1776,
 460,
 739,
 778,
 836,
 965,
 485,
 159,
 264,
 760,
 808,
 1162,
 990,
 478,
 221,
 1606,
 1550,
 431,
 710,
 212,
 635,
 1577,
 578,
 8,
 203,
 796,
 919,
 1682,
 10,
 1700,
 1068,
 918,
 1292,
 1293,
 1479,
 1574,
 1095,
 394,
 1718,
 795,
 1132,
 1620,
 129,
 1906,
 1034,
 175,
 1405,
 1621])

In [ ]:
not_vert = not vert_lev[comp][var_ind]

In [ ]:
slice_numbers = np.arange(1,52)

In [ ]:
%%time

# If 2D variable that isn't RESTOM
if var != 'RESTOM' and not_vert:
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0

    # Loop through all start years
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        # If the 75-year slice is contained within a single 100-year long file (ex: 710-784, within 700-799 file)
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            outpath = path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr'
            if os.path.isdir(outpath):
                print(cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'zarr exists')
                continue
            else:
                # Open dataset
                filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn['in'][freq]+'nc'
                print(filepath+filename)
                ds = dask.delayed(xr.open_dataset)(filepath+filename,chunks=chunks[comp])
        
        # Else 75-year slice is split across two 100-year files (ex: 795-869, need to load both 700-799 and 800-899 files)
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            # Open dataset
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn2['in'][freq]+'nc'
            print(filepath+filename1)
            print(filepath+filename2)
            ds = dask.delayed(xr.open_mfdataset)([filepath+filename1, filepath+filename2],chunks=chunks[comp])

        
        dsv = ds[var]
     
        # Process each year in range(start year, end year)
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr_sl = yr_range[i]
                endyr_sl = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
                
                fixedtime_data = FixTime(ann_slice)
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ice','gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                elif comp == 'ocn':
                    if var == 'TEMP':
                        fixedtime_data = fixedtime_data.sel(z_t=0,method='nearest')
                        fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'ocn','gx1v7')
                        processed_list.append(fixedgrid_data)
                        print('   fixed CICE grid')
                    else:
                        processed_list.append(fixedtime_data)

                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
                    
        
        if freq == 0 and not_vert:
            processed_comp = dask.compute(*processed_list)
            print('computed list')
            
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

In [ ]:
%%time

# If 3D variable
if var != 'RESTOM' and vert_lev[comp][var_ind]:
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0
    slice_list = []

    # Loop through all start years
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        # If the 75-year slice is contained within a single 100-year long file (ex: 710-784, within 700-799 file)
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            
            # Open dataset
            filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn['in'][freq]+'nc'
            print(filepath+filename)
            ds = xr.open_dataset(filepath+filename,chunks=chunks[comp])

        # Else 75-year slice is split across two 100-year files (ex: 795-869, need to load both 700-799 and 800-899 files)
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}

            # Open dataset
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+var+yr_extn2['in'][freq]+'nc'
            print(filepath+filename1)
            print(filepath+filename2)
            ds = xr.open_mfdataset([filepath+filename1, filepath+filename2],chunks=chunks[comp])
        
        
        dsv = ds

        # Add pressure variable to dataset
        if start_floor == end_floor:
            filename = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn['in'][freq]+'nc'
            dsp = xr.open_dataset(filepath+filename,chunks=chunks[comp])
        else:
            filename1 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn1['in'][freq]+'nc'
            filename2 = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'PS'+yr_extn2['in'][freq]+'nc'
            dsp = xr.open_mfdataset([filepath+filename1, filepath+filename2],chunks=chunks[comp])
            
        dsv['PS'] = dsp['PS']
        
        # Process each year in range(start year, end year)
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # Selection of final time
            final_time = pd.date_range(str(1950+i)+'-01-01',str(1950+i)+'-12-01',freq='MS')
                
            # Selection of data    
            startyr_sl = yr_range[i]
            endyr_sl = yr_range[i+1]
            ann_slice = dsv.sel(time=slice(startyr_sl+'-02-01',endyr_sl+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
            print('sliced '+startyr_sl+'-02-01 to '+endyr_sl+'-01-17')
            
            ann_slice = FixTime(ann_slice)
            ann_slice = ann_slice.assign_coords(time=final_time)
            print('   fixed time')

            ann_slice = FixLongitude(ann_slice, False)
            print('   fixed longitude')
            
            ann_slice = InterPlevels(ann_slice, var)
            print('   interpolated p-levels')
            processed_list.append(ann_slice)

        
        processed_out = xr.concat(processed_list,dim='time').chunk({'time':111})
        processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
        slice_list.append(processed_out)  
        print('concatenated slice '+str(num))
        num = num+1
        
    for i in range(0,len(slice_list)):
        if i == 0:
            slice_list[i].to_zarr(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+'.195001-202412.zarr', 
                            group=var)
            print('saved initial zarr store')
        else:
            print('saving slice '+str(i+1))
            slice_list[i].to_zarr(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+'.195001-202412.zarr', 
                                    append_dim='slice', mode='a-',group=var)
    print('wrote data to disk')
    
       

In [ ]:
%%time

# If variable is RESTOM (special because need to open two different variables to calculate RESTOM)
if var == 'RESTOM':
    filepath = path_to_arch+cesm2piC+'/'+comp+'/proc/tseries/'+time_path[comp][freq]+'/'
    num = 0

    # Loop through start years
    for startyr in startyrs:
        yr_range = np.array([str(i).zfill(4) for i in np.arange(startyr,startyr+76)])
        
        endyr = startyr+74
        start_floor = np.floor_divide(startyr,100)
        end_floor = np.floor_divide(endyr,100)
        
        # If the 75-year slice is contained within a single 100-year long file (ex: 710-784, within 700-799 file)
        if start_floor == end_floor:
            if start_floor == 0:
                instart = str(1).zfill(4)
                inend = str(99).zfill(4)
            elif start_floor == 19:
                instart = str(1900).zfill(4)
                inend = str(2000).zfill(4)
            else:
                instart = str(start_floor*100).zfill(4)
                inend = str(start_floor*100+99).zfill(4)
        
            yr_extn = {'in': ["."+instart+"01-"+inend+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
        
            # Open dataset
            filename_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn['in'][freq]+'nc'
            filename_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn['in'][freq]+'nc'
            print(filepath+filename_fsnt)
            ds_fsnt = dask.delayed(xr.open_dataset)(filepath+filename_fsnt,chunks=chunks[comp])
            ds_flnt = dask.delayed(xr.open_dataset)(filepath+filename_flnt,chunks=chunks[comp])

            ds_fsnt = ds_fsnt['FSNT']
            ds_flnt = ds_flnt['FLNT']

            dsv = ds_fsnt-ds_flnt
            dsv = dsv.rename(var) 
            
        # Else 75-year slice is split across two 100-year files (ex: 795-869, need to load both 700-799 and 800-899 files)
        else:
            if start_floor == 0:
                instart1 = str(1).zfill(4)
                inend1 = str(99).zfill(4)
            else:
                instart1 = str(start_floor*100).zfill(4)
                inend1 = str(start_floor*100+99).zfill(4)
                
            if end_floor == 19:
                instart2 = str(1900).zfill(4)
                inend2 = str(2000).zfill(4)
            else:
                instart2 = str(end_floor*100).zfill(4)
                inend2 = str(end_floor*100+99).zfill(4)
    
            yr_extn1 = {'in': ["."+instart1+"01-"+inend1+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            yr_extn2 = {'in': ["."+instart2+"01-"+inend2+"12.", ".05000101-05991231."],
                       'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
            yr_extn = {'out': ["."+str(startyr).zfill(4)+"01-"+str(endyr).zfill(4)+"12.", ".05010101-05741231."]}
    
            # Open dataset
            filename1_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn1['in'][freq]+'nc'
            filename2_fsnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FSNT'+yr_extn2['in'][freq]+'nc'

            filename1_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn1['in'][freq]+'nc'
            filename2_flnt = cesm2piC+'.'+mod_com[comp]+h_ext[comp][freq]+'FLNT'+yr_extn2['in'][freq]+'nc'
            
            print(filepath+filename1_fsnt)
            print(filepath+filename2_fsnt)
            ds_fsnt = dask.delayed(xr.open_mfdataset)([filepath+filename1_fsnt, filepath+filename2_fsnt],chunks=chunks[comp])
            ds_flnt = dask.delayed(xr.open_mfdataset)([filepath+filename1_flnt, filepath+filename2_flnt],chunks=chunks[comp])

            ds_fsnt = ds_fsnt['FSNT']
            ds_flnt = ds_flnt['FLNT']

            dsv = ds_fsnt-ds_flnt
            dsv = dsv.rename(var)
        
        # Process each year in range(start year, end year)
        processed_list = []
        for i in range(0,len(yr_range)-1):
            # If monthly data
            if freq == 0:
                startyr = yr_range[i]
                endyr = yr_range[i+1]
                ann_slice = dsv.sel(time=slice(startyr+'-02-01',endyr+'-01-17')) #if add_cyclic else dsv.sel(time=slice(startyr+'-02-10',endyr+'-01-17'),lat=slice(0,90))
                print('sliced '+startyr+'-02-01 to '+endyr+'-01-17')
                
                fixedtime_data = dask.delayed(FixTime)(ann_slice)
                print('   fixed time')
        
                if comp == 'ice':
                    fixedgrid_data = dask.delayed(FixGrid)(fixedtime_data,'gx1v7')
                    processed_list.append(fixedgrid_data)
                    print('   fixed CICE grid')
        
                elif comp == 'ocn':
                    processed_list.append(fixedtime_data)
        
                else:
                    fixedgrid_data = dask.delayed(FixLongitude)(fixedtime_data, False)
                    processed_list.append(fixedgrid_data)
                    print('   fixed longitude')
        
        
        
        processed_comp = dask.compute(*processed_list)
        print('computed list')
        
        if not_vert and freq == 0:
            processed_out = xr.concat(processed_comp,dim='time').chunk({'time':111})
            print('concatenated data')
    
            processed_out = processed_out.expand_dims(slice=[slice_numbers[num]])
            
            processed_out.to_netcdf(path_to_outdata+cesm2piC+h_ext[comp][freq]+var+yr_extn['out'][freq]+'nc', 
                                        format='NETCDF4',encoding={var: {"zlib": True, "complevel": 1}})
            print('wrote data to disk')
    
        num = num+1

In [ ]:
client.shutdown()